# Proposal model v2 (M12e): train on the OMR pool, on Colab

Checklist and evaluation plan: `results/m12e_training_plan.md`. In short:

1. On the PC, export the tokens (`python -m brickwfc.experiments.export_tokens --pool results/m12e_omr_pool.json --out data/tokens_v2 --results results/m12e_tokens.md`) and upload `data/tokens_v2/` to Google Drive as `MyDrive/brick-wfc/tokens_v2/`.
2. Colab: **Runtime → Change runtime type → GPU** (A100 or L4 on Colab Pro; T4 works with `batch=32`). Add a secret **`GITHUB_TOKEN`** (key icon on the left; a fine-grained token with read access to the repository) and give this notebook access to it.
3. Set `CONFIG` below: run **`smoke`** first (a few minutes; checks every step end to end), then `base` and `scaled`.
4. **Runtime → Run all.** Every cell is safe to rerun. After a disconnect, run all again: training resumes from the last checkpoint on Drive.
5. The run folder on Drive (`MyDrive/brick-wfc/runs/<run>/`: `proposal.npz`, `vocab.json`, `log.jsonl`, `checkpoint.pt`) is the result; copy it to `data/models/` on the PC for the build evaluation.

In [ ]:
# Settings
CONFIG = 'smoke'        # 'smoke' (minutes), 'base' (5.2M params, M12b's size on the pool), 'scaled' (~26M params)
BRANCH = 'main'         # the branch or commit whose catalog the tokens were exported with
REPO = 'github.com/macint0/brick-wfc.git'
DRIVE = '/content/drive/MyDrive/brick-wfc'
TOKENS = 'tokens_v2'    # folder under DRIVE: vocab.json, export.json, train/val/setval/heldout.npz
SET_SHARE = None        # share of training windows from real sets; None = their share of training parts (export.json)
SEED = 0

CONFIGS = {
    # steps is an upper bound; with patience > 0, early stopping on the selection loss usually ends the run first
    'smoke':  dict(d_model=128, layers=2, heads=4, steps=300, batch=16, lr=1e-3, warmup=50, eval_every=100,
                   val_builds=50, patience=0, minutes=10),
    'base':   dict(d_model=256, layers=6, heads=8, steps=40000, batch=64, lr=5e-4, warmup=500, eval_every=1000,
                   val_builds=400, patience=5, minutes=0),
    'scaled': dict(d_model=512, layers=8, heads=8, steps=60000, batch=64, lr=3e-4, warmup=1000, eval_every=1000,
                   val_builds=400, patience=5, minutes=0),
}
C = CONFIGS[CONFIG]
RUN = f'{CONFIG}_seed{SEED}'
OUT = f'{DRIVE}/runs/{RUN}'
print(CONFIG, C, '->', OUT)

In [ ]:
# Google Drive (tokens and checkpoints live there) and the GPU
from google.colab import drive
drive.mount('/content/drive')
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Clone the repository (private: the token comes from the GITHUB_TOKEN secret and is not stored in the clone)
import os, subprocess
from google.colab import userdata
token = userdata.get('GITHUB_TOKEN')
url = f'https://x-access-token:{token}@{REPO}'
if not os.path.exists('/content/brick-wfc/.git'):
    subprocess.run(['git', 'clone', '--quiet', url, '/content/brick-wfc'], check=True)
    subprocess.run(['git', '-C', '/content/brick-wfc', 'remote', 'set-url', 'origin', f'https://{REPO}'], check=True)
subprocess.run(['git', '-C', '/content/brick-wfc', 'fetch', '--quiet', url, BRANCH], check=True)
subprocess.run(['git', '-C', '/content/brick-wfc', 'checkout', '--quiet', 'FETCH_HEAD'], check=True)
del token, url
os.chdir('/content/brick-wfc')
!git log -1 --format='%h %ad %s' --date=short

In [ ]:
# Install the package (numpy, scipy, Pillow, trimesh) and check PyTorch. Colab ships a CUDA build of torch;
# one is installed only if it is missing (cu124 wheels, as on the PC). torch is not a dependency of the package.
!pip install --quiet -e .
try:
    import torch
except ImportError:
    !pip install --quiet torch --index-url https://download.pytorch.org/whl/cu124
    import torch
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(), '|',
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'no GPU: Runtime -> Change runtime type')

In [ ]:
# Tokens: copy from Drive to the local disk (fast reads) and check they match the checked-out catalog
import json, shutil
from brickwfc.experiments.omr_pool import catalog_fingerprint
shutil.copytree(f'{DRIVE}/{TOKENS}', '/content/tokens', dirs_exist_ok=True)
export = json.load(open('/content/tokens/export.json'))
print(json.dumps(export, indent=1))
assert export['catalog'] == catalog_fingerprint(), (
    f"tokens were exported with catalog {export['catalog']}, this checkout has {catalog_fingerprint()}: "
    'set BRANCH to the commit the tokens were exported from, or export again')
share = SET_SHARE if SET_SHARE is not None else export['set_share_of_train_parts']
print('set share', share)

In [ ]:
# Train. Resumable: rerunning this cell (or Run all) after a disconnect picks up the checkpoint on Drive.
# The best weights by the selection loss (mean of BrickGPT val and the pool's setval) are exported as proposal.npz.
args = (f"--data /content/tokens --out {OUT} --d-model {C['d_model']} --layers {C['layers']} --heads {C['heads']} "
        f"--steps {C['steps']} --batch {C['batch']} --lr {C['lr']} --warmup {C['warmup']} "
        f"--eval-every {C['eval_every']} --val-builds {C['val_builds']} --patience {C['patience']} "
        f"--minutes {C['minutes']} --set-share {share} --seed {SEED}")
print(args)
!python training/train.py {args}

In [ ]:
# The numpy inference used on the CPU must match PyTorch on the trained weights
!PYTHONPATH=src python training/parity.py check --weights {OUT}/proposal.npz --data /content/tokens

In [ ]:
# Losses per evaluation: train, BrickGPT val, pool setval (selection = mean of the two), held-out demo sets (reported only)
rows = [json.loads(line) for line in open(f'{OUT}/log.jsonl')]
best = min(rows, key=lambda r: r['select'])
print(f"{'step':>6} {'min':>6} {'train':>7} {'val':>7} {'setval':>7} {'select':>7} {'heldout':>8}  heldout shape / color")
for r in rows:
    print(f"{r['step']:>6} {r['minutes']:>6.1f} {r['train_loss']:>7.3f} {r['val']['loss']:>7.3f} "
          f"{r.get('set_val', {}).get('loss', float('nan')):>7.3f} {r['select']:>7.3f} {r['heldout_sets']['loss']:>8.3f}  "
          f"{r['heldout_sets']['shape']:.3f} / {r['heldout_sets']['color']:.3f}{'  <- best' if r is best else ''}")

## After the run

- Copy `MyDrive/brick-wfc/runs/<run>/` to `data/models/proposal_v2_<run>/` on the PC (`proposal.npz`, `vocab.json`, `log.jsonl`; the checkpoint is not needed there).
- Record the run (config, GPU, minutes, best step, the loss table above) in `results/m12e_training.md`.
- The build evaluation (held-out demo sets, looks metrics, rendered sheets) is in `results/m12e_training_plan.md`.